# Conversación A2A: agente_alejandra ↔ agente_diego_vasquez

Mi agente corre en `http://127.0.0.1:2025`. El de Diego llega por un túnel de Cloudflare (la URL cambia si él reinicia el túnel).

In [ ]:
# ── 1. CONFIGURACIÓN Y DESCUBRIMIENTO ────────────────────────────────────
import httpx
from langgraph_sdk import get_client
from a2a.client import A2ACardResolver, ClientConfig, create_client

URL_ALEJANDRA = "http://127.0.0.1:2025"
URL_DIEGO = "https://dangerous-travels-certified-countries.trycloudflare.com"
SERVIDORES = [URL_ALEJANDRA, URL_DIEGO]

# Libera las conexiones si ejecutas el bloque otra vez.
for cliente in globals().get("clientes_a2a", {}).values():
    await cliente.close()
if "http_a2a" in globals():
    await http_a2a.aclose()

http_a2a = httpx.AsyncClient(timeout=180, trust_env=False)
config_a2a = ClientConfig(
    httpx_client=http_a2a,
    streaming=False,
    supported_protocol_bindings=["JSONRPC"],
)

agentes_por_nombre, clientes_a2a = {}, {}

for base in SERVIDORES:
    base = base.rstrip("/")
    try:
        async with get_client(url=base, api_key=None, timeout=30) as remoto:
            lista = await remoto.assistants.search(limit=100, response_format="array")
    except Exception as error:
        print(f"No pude consultar {base}: {type(error).__name__}")
        continue
    resolver = A2ACardResolver(http_a2a, base)
    for asistente in lista:
        aid = asistente["assistant_id"]
        card = await resolver.get_agent_card(
            http_kwargs={"params": {"assistant_id": aid}}
        )
        # La tarjeta remota anuncia su dirección local; usamos la que nos sirve.
        for interfaz in card.supported_interfaces:
            if interfaz.protocol_binding == "JSONRPC":
                interfaz.protocol_version = "0.3.0"
                interfaz.url = f"{base}/a2a/{aid}"
        if card.name in agentes_por_nombre:
            raise ValueError(f"Nombre repetido: {card.name}. Cambia el nombre de la tarjeta.")
        clientes_a2a[aid] = await create_client(card, client_config=config_a2a)
        agentes_por_nombre[card.name] = aid
        print(f"{card.name} ← {base}\n  {card.description[:150]}")

print("\nAgentes disponibles:", list(agentes_por_nombre))

In [ ]:
# ── 2. FUNCIÓN PARA PREGUNTAR POR A2A ────────────────────────────────────
from a2a.helpers import new_text_message, get_stream_response_text, get_message_text
from a2a.types import Role, SendMessageRequest

async def preguntar(cliente, texto, contexto=None):
    peticion = SendMessageRequest(message=new_text_message(
        texto, role=Role.ROLE_USER, context_id=contexto
    ))
    partes = []
    async for evento in cliente.send_message(peticion):
        fragmento = get_stream_response_text(evento)
        if evento.HasField("task"):
            contexto = evento.task.context_id or contexto
            if not fragmento and evento.task.status.HasField("message"):
                fragmento = get_message_text(evento.task.status.message)
        elif evento.HasField("message"):
            contexto = evento.message.context_id or contexto
        if fragmento:
            partes.append(fragmento)
    if not partes:
        raise RuntimeError("El agente no devolvió texto. Revisa su terminal.")
    return "\n".join(partes), contexto

# Nombres exactos de la lista «Agentes disponibles».
NOMBRE_A = "agente_alejandra"
NOMBRE_B = "agente_diego_vasquez"
A = clientes_a2a[agentes_por_nombre[NOMBRE_A]]
B = clientes_a2a[agentes_por_nombre[NOMBRE_B]]

In [ ]:
# ── 3. PRESENTACIÓN: ¿QUIÉN ERES? ────────────────────────────────────────
presentacion_b, _ = await preguntar(B, "Hola, soy el agente de Alejandra. ¿Quién eres y en qué me puedes ayudar?")
print(f"{NOMBRE_B}:\n{presentacion_b}")

In [ ]:
# ── 4. CONVERSACIÓN DE TRES TURNOS ───────────────────────────────────────
# Cambia el reto por uno que necesite las dos especialidades.
reto = ("Queremos un pipeline de datos que cruce información del SII y de la CMF para detectar, "
        "antes de que sean un problema, atrasos tributarios de empresas y concentración por emisor "
        "en fondos de inversión (límite del 20% de la NCG 365). Necesitamos un diseño conjunto: "
        "qué reglas de auditoría aplicar y cómo construir el pipeline.")

# Turno 1: A propone
aporte_a, contexto_a = await preguntar(
    A, f"Reto: {reto} Propón tres ideas desde tu especialidad."
)
# Turno 2: B mejora
aporte_b, contexto_b = await preguntar(
    B, f"Reto: {reto} Tu colega propone: {aporte_a}\n"
    "Mejora su propuesta desde tu especialidad y explica qué cambiarías."
)
# Turno 3: A integra
acuerdo, contexto_a = await preguntar(
    A, f"Tu colega responde: {aporte_b}\n"
    "Integra ambos aportes en un plan conjunto breve.",
    contexto_a,
)

for nombre, texto in [(NOMBRE_A, aporte_a), (NOMBRE_B, aporte_b), (NOMBRE_A, acuerdo)]:
    print(f"\n{nombre}:\n{texto}")

In [ ]:
# Cierre: liberar conexiones.
for cliente in clientes_a2a.values():
    await cliente.close()
await http_a2a.aclose()
print("Conexiones cerradas.")